In [91]:
from pathlib import Path 
import math
import pickle
import bisect
import numpy as np
import time

In [92]:
DEBUG = True
W_SIZE = 1000
AREA_MAX = 1000 * 1000
AREA_TH = 0.996
INF = 10 ** 18

def debug_print(*args, **kwargs):
    if DEBUG:
        print(*args, **kwargs)

In [93]:
def fastcopy(obj):
    return pickle.loads(pickle.dumps(obj, -1))

class Env:
    def __init__(self, input_txt_path: Path):
        self.W, self.D, self.N, self.a = self._input(input_txt_path)

    def _input(self, txt_path):
        with open(txt_path, mode="r") as file:
            lines = file.readlines()
        W, D, N = map(int, lines[0].split())
        a = []
        for line, d in zip(lines[1:], range(D)):
            d = list(map(int, line.split()))
            a.append(d)
        return W, D, N, a

class Area:
    def __init__(self, id_val: int, self_area: int, target_area: int):
        self.id = id_val
        self.self_area = self_area
        self.target_area = target_area
        self.need_first = False

In [94]:
def check_ans(ans: list[tuple[int]]):
    for d in range(len(ans)):
        for coordinates in ans[d]:
            x1, y1, x2, y2 = coordinates
            if x1 < 0 or x2 > 1000 or y1 < 0 or y2 > 1000:
                return False
    return True

In [95]:
def calc_cost(ans: list[list[int]], env: Env):
    partial_cost = 0
    area_cost = 0

    hs = set()
    vs = set()
    for d in range(env.D):
        hs2 = set()
        vs2 = set()
        for k in range(env.N):
            i0, j0, i1, j1 = ans[d][k]
            area = (i1 - i0) * (j1 - j0)
            if env.a[d][k] > area:
                area_cost += 100 * (env.a[d][k] - area)
            for j in range(j0, j1):
                if i0 > 0:
                    hs2.add((i0, j))
                if i1 < env.W:
                    hs2.add((i1, j))
            for i in range(i0, i1):
                if j0 > 0:
                    vs2.add((j0, i))
                if j1 < env.W:
                    vs2.add((j1, i))

        if d > 0:
            partial_cost += len(hs ^ hs2)
            partial_cost += len(vs ^ vs2)

        hs = hs2
        vs = vs2
    return partial_cost + area_cost + 1

In [96]:
def dicision_pos(one_day_area: list[tuple[int, int]], now_lr, now_ud, x1, y1, x2, y2, how=None):
    area_diffs = []
    for i, area in one_day_area:
        len1 = math.ceil(area / now_lr)
        len2 = math.ceil(area / now_ud)
        if len1 * now_lr < len2 * now_ud:
            direct = "ud"
            if how == "diff":
                score = len1 * now_lr - area
            elif how == "aspect":
                score = 1 - min(len1, now_lr) / max(len1, now_lr)
            area_diffs.append((score, len1, direct, i))
        else:
            direct = "lr"
            if how == "diff":
                score = len2 * now_ud - area
            elif how == "aspect":
                score = 1 - min(len2, now_ud) / max(len2, now_ud)
            area_diffs.append((score, len2, direct, i))
    _, min_len, min_direct, min_i = min(area_diffs)
    if min_direct == "ud":
        ans = (min_i, x1, y1, x2, y1 + min_len)
        assigin_area = (x2 - x1) * min_len
        y1 += min_len
    elif min_direct == "lr":
        ans = (min_i, x2 - min_len, y1, x2, y2)
        assigin_area = min_len * (y2 - y1)
        x2 -= min_len
    else:
        raise ValueError("invalid direct")
    
    for i, area in one_day_area:
        if i == min_i:
            target_area = area

    remain_diff_area = assigin_area - target_area

    return x1, y1, x2, y2, ans, remain_diff_area

In [97]:
def one_day_greedy_solve(one_day_area: list[int], env: Env):
    x1 = 0
    y1 = 0
    x2 = W_SIZE
    y2 = W_SIZE
    remain_area = W_SIZE * W_SIZE - sum(one_day_area)

    ans = [None for _ in range(env.N)]
    one_day_area_with_index = [(i, area) for i, area in enumerate(one_day_area)]
    for n in range(env.N):
        remain_n = env.N - n
        now_lr = x2 - x1
        now_ud = y2 - y1
        if n == env.N - 1:
            # 最後の一個は残り全部
            last_ind = one_day_area_with_index[0][0]
            ans[last_ind] = (x1, y1, x2, y2)
            break

        if remain_n * (W_SIZE - 1) <= remain_area:
            # 残り面積が広いときはアスペクト比を貪欲探索
            x1, y1, x2, y2, now_ans, assigin_area = dicision_pos(one_day_area_with_index, now_lr, now_ud, x1, y1, x2, y2, how="aspect")
        else:
            # 残り面積が狭いときは面積を有効活用
            x1, y1, x2, y2, now_ans, assigin_area = dicision_pos(one_day_area_with_index, now_lr, now_ud, x1, y1, x2, y2, how="diff")
        ans_ind, *ans_tuple = now_ans

        for i, area in one_day_area_with_index:
            if i == ans_ind:
                one_day_area_with_index.remove((i, area))
                
        ans[ans_ind] = ans_tuple
        remain_area -= assigin_area
    
    return ans

In [98]:
def solve(env: Env):
    ans = []
    for d in range(env.D):
        ans.append(one_day_greedy_solve(env.a[d], env))
    cost = calc_cost(ans, env)

    debug_print(f"cost:{cost}")
    return ans

In [99]:
def main():
    for i in range(100):
        debug_print(f"----- {i}/100 -----")
        input_txt_path = Path(f"./in/{str(i).zfill(4)}.txt")
        output_txt_path = Path(f"./out/{str(i).zfill(4)}.txt")
        env = Env(input_txt_path)

        ans = solve(env)
        if not check_ans(ans):
            raise ValueError()

        str_ans = []
        for d in range(env.D):
            for k in range(env.N):
                i0, j0, i1, j1 = ans[d][k]
                str_ans.append(f"{i0} {j0} {i1} {j1}")

        with open(output_txt_path, mode="w") as file:
            file.write("\n".join(str_ans))
main()

----- 0/100 -----
cost:47578
----- 1/100 -----
cost:1014321
----- 2/100 -----
cost:342604
----- 3/100 -----
cost:413040
----- 4/100 -----
cost:305059
----- 5/100 -----
cost:1325033
----- 6/100 -----
cost:133685
----- 7/100 -----
cost:423931
----- 8/100 -----
cost:457777
----- 9/100 -----
cost:59137
----- 10/100 -----
cost:182040
----- 11/100 -----
cost:503809
----- 12/100 -----
cost:964943
----- 13/100 -----
cost:498034
----- 14/100 -----
cost:830786
----- 15/100 -----
cost:705435
----- 16/100 -----
cost:777211
----- 17/100 -----
cost:604679
----- 18/100 -----
cost:1277111
----- 19/100 -----
cost:1313664
----- 20/100 -----
cost:523558
----- 21/100 -----
cost:740755
----- 22/100 -----
cost:769608
----- 23/100 -----
cost:717187
----- 24/100 -----
cost:1081289
----- 25/100 -----
cost:1383978
----- 26/100 -----
cost:161038
----- 27/100 -----
cost:1185940
----- 28/100 -----
cost:664809
----- 29/100 -----
cost:1022823
----- 30/100 -----
cost:282612
----- 31/100 -----
cost:217763
----- 32/100